# Signal Concatenate Verification

This notebook verifies that the signal concatenate logic in `group_signals_by_position()`
correctly reconstructs the original signal from BLOW5 files.

## Method

1. Run eventalign on a sample (or use existing eventalign TSV output)
2. Select multiple reads at multiple positions
3. Use our `group_signals_by_position()` to concatenate signals for each (read, position)
4. Record the min `start_idx` and max `end_idx` for each (read, position) pair
5. Read the original BLOW5 file and extract raw signals for those reads
6. Slice the raw signal using `[start_idx:end_idx]`
7. Compare the concatenated signal with the sliced raw signal

## Dependencies

This notebook requires `pyslow5`:

```bash
pip install pyslow5
```

Or use slow5tools CLI:

```bash
conda install -c bioconda slow5tools
```

## 0. Setup

In [ ]:
import logging
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np

# Enable INFO-level logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)

# Check pyslow5 availability
try:
    import pyslow5
    PYSLOW5_AVAILABLE = True
    print(f"pyslow5 available: version {pyslow5.__version__ if hasattr(pyslow5, '__version__') else 'unknown'}")
except ImportError:
    PYSLOW5_AVAILABLE = False
    print("WARNING: pyslow5 not installed. Install with: pip install pyslow5")
    print("Falling back to slow5tools CLI (requires slow5tools on PATH)")

print(f"numpy version: {np.__version__}")

## 1. Configuration

Set paths to your eventalign TSV output and BLOW5 files.

In [ ]:
# Adjust these paths to point to your data
TESTDATA = Path("../testdata").resolve()

# Eventalign TSV output (must have --signal-index flag to include start_idx/end_idx)
EVENTALIGN_TSV = TESTDATA / "output" / "eventalign" / "native_eventalign.tsv"

# Original BLOW5 file
BLOW5_FILE = TESTDATA / "native_0" / "blow5" / "nanopore.blow5"

# Verify files exist
print(f"Eventalign TSV: {EVENTALIGN_TSV} ({'EXISTS' if EVENTALIGN_TSV.exists() else 'MISSING'})")
print(f"BLOW5 file:     {BLOW5_FILE} ({'EXISTS' if BLOW5_FILE.exists() else 'MISSING'})")

if not EVENTALIGN_TSV.exists():
    print("\nWARNING: Eventalign TSV not found.")
    print("Please run eventalign with --signal-index flag first:")
    print("  f5c eventalign -b native.bam -r reads.fq -g ref.fa --slow5 nanopore.blow5 \\")
    print("    --samples --signal-index --scale-events --print-read-names > eventalign.tsv")

## 2. Parse Eventalign and Group Signals

Use `group_signals_by_position()` to concatenate signals, while also tracking
the min `start_idx` and max `end_idx` for each (read, position) pair.

In [ ]:
from baleen.eventalign._signal import parse_eventalign, EventalignRow

# Custom grouping that tracks start_idx and end_idx ranges
def group_signals_with_indices(tsv_path: Path) -> dict:
    """
    Group signals by (read_name, position) while tracking start_idx and end_idx ranges.
    
    Returns
    -------
    dict[int, dict]
        {position: {
            'contig': str,
            'reference_kmer': str,
            'reads': {
                read_name: {
                    'signal': np.ndarray,        # concatenated signal
                    'start_idx': int,            # minimum start_idx
                    'end_idx': int,              # maximum end_idx
                    'n_events': int,             # number of events merged
                }
            }
        }}
    """
    # {(position, read_name): [(start_idx, end_idx, samples), ...]}
    pending: dict[tuple[int, str], list[tuple[int | None, int | None, np.ndarray]]] = defaultdict(list)
    
    # Track contig and kmer info per position
    position_info: dict[int, tuple[str, str]] = {}
    
    for event in parse_eventalign(tsv_path):
        key = (event.position, event.read_name)
        pending[key].append((event.start_idx, event.end_idx, event.samples))
        
        if event.position not in position_info:
            position_info[event.position] = (event.contig, event.reference_kmer)
    
    # Build result structure
    result: dict[int, dict] = {}
    
    for (position, read_name), events in pending.items():
        if position not in result:
            contig, kmer = position_info[position]
            result[position] = {
                'contig': contig,
                'reference_kmer': kmer,
                'reads': {},
            }
        
        # Separate events with and without indices
        with_idx = [(s, e, sig) for s, e, sig in events if s is not None and e is not None]
        without_idx = [(s, e, sig) for s, e, sig in events if s is None or e is None]
        
        # Sort by start_idx
        with_idx.sort(key=lambda x: x[0])
        
        # Concatenate signals
        all_signals = [sig for _, _, sig in with_idx] + [sig for _, _, sig in without_idx]
        concatenated = np.concatenate(all_signals).astype(np.float32)
        
        # Track index range
        if with_idx:
            min_start = min(s for s, _, _ in with_idx)
            max_end = max(e for _, e, _ in with_idx)
        else:
            min_start = None
            max_end = None
        
        result[position]['reads'][read_name] = {
            'signal': concatenated,
            'start_idx': min_start,
            'end_idx': max_end,
            'n_events': len(events),
        }
    
    return result

# Parse eventalign
if EVENTALIGN_TSV.exists():
    print(f"Parsing eventalign TSV: {EVENTALIGN_TSV}")
    grouped = group_signals_with_indices(EVENTALIGN_TSV)
    print(f"Found {len(grouped)} positions")
    
    # Count total reads
    total_reads = sum(len(p['reads']) for p in grouped.values())
    print(f"Total (position, read) pairs: {total_reads}")
    
    # Count reads with valid indices
    with_indices = sum(
        1 for p in grouped.values() 
        for r in p['reads'].values() 
        if r['start_idx'] is not None
    )
    print(f"Pairs with valid start_idx/end_idx: {with_indices}")
else:
    grouped = {}
    print("Eventalign TSV not found, skipping parsing.")

## 3. Select Test Cases

Select a few (read, position) pairs to verify. We'll pick reads with:
- Multiple events at the same position (to test concatenate logic)
- Valid start_idx and end_idx

In [ ]:
# Find test cases: (position, read_name) pairs with multiple events and valid indices
test_cases = []

for pos, pos_data in grouped.items():
    for read_name, read_data in pos_data['reads'].items():
        if read_data['start_idx'] is not None and read_data['n_events'] >= 2:
            test_cases.append({
                'position': pos,
                'read_name': read_name,
                'start_idx': read_data['start_idx'],
                'end_idx': read_data['end_idx'],
                'n_events': read_data['n_events'],
                'signal_length': len(read_data['signal']),
                'concatenated_signal': read_data['signal'],
            })

# Sort by number of events (more events = better test)
test_cases.sort(key=lambda x: x['n_events'], reverse=True)

print(f"Found {len(test_cases)} test cases with multiple events and valid indices")
print()

# Show top 10 test cases
if test_cases:
    print("Top 10 test cases (sorted by n_events):")
    print(f"{'Position':>10}  {'Read':<30}  {'Events':>6}  {'start_idx':>10}  {'end_idx':>10}  {'Signal len':>10}")
    print("-" * 90)
    for tc in test_cases[:10]:
        print(f"{tc['position']:10d}  {tc['read_name'][:30]:<30}  {tc['n_events']:6d}  "
              f"{tc['start_idx']:10d}  {tc['end_idx']:10d}  {tc['signal_length']:10d}")
    
    # Select up to 5 test cases for verification
    SELECTED_CASES = test_cases[:5]
else:
    SELECTED_CASES = []
    print("No suitable test cases found.")

## 4. Read Raw Signals from BLOW5

Use pyslow5 to read the original raw signals for the selected reads.

In [ ]:
def read_raw_signals_pyslow5(blow5_path: Path, read_names: list[str]) -> dict[str, np.ndarray]:
    """
    Read raw signals for specified reads from a BLOW5 file using pyslow5.
    
    Parameters
    ----------
    blow5_path : Path
        Path to BLOW5 file
    read_names : list[str]
        List of read names to extract
    
    Returns
    -------
    dict[str, np.ndarray]
        {read_name: raw_signal} where raw_signal is float32 pA values
    """
    if not PYSLOW5_AVAILABLE:
        raise ImportError("pyslow5 not installed")
    
    s5 = pyslow5.Open(str(blow5_path), 'r')
    
    raw_signals = {}
    read_set = set(read_names)
    
    # Read all reads and filter
    reads, _ = s5.read_read_batch(True, read_set)
    
    for read_name, read_data in reads.items():
        # 'signal' is the raw signal in pA (float32)
        raw_signals[read_name] = np.array(read_data['signal'], dtype=np.float32)
    
    s5.close()
    return raw_signals


def read_raw_signals_cli(blow5_path: Path, read_names: list[str], tmp_dir: Path) -> dict[str, np.ndarray]:
    """
    Read raw signals using slow5tools CLI as fallback.
    """
    import subprocess
    import tempfile
    
    # Write read names to temp file
    tmp_dir.mkdir(parents=True, exist_ok=True)
    reads_file = tmp_dir / "read_names.txt"
    with reads_file.open("w") as f:
        for name in read_names:
            f.write(name + "\n")
    
    # Extract reads to temp BLOW5
    tmp_blow5 = tmp_dir / "extracted.blow5"
    cmd = ["slow5tools", "get", "--list", str(reads_file), str(blow5_path), "-o", str(tmp_blow5)]
    subprocess.run(cmd, check=True, capture_output=True)
    
    # Parse the extracted BLOW5 (would need pyslow5 anyway)
    # This is a fallback that requires more work
    raise NotImplementedError("slow5tools CLI fallback not fully implemented - please install pyslow5")


# Read raw signals for selected test cases
if SELECTED_CASES and BLOW5_FILE.exists():
    unique_reads = list(set(tc['read_name'] for tc in SELECTED_CASES))
    print(f"Reading raw signals for {len(unique_reads)} unique reads from BLOW5...")
    
    try:
        if PYSLOW5_AVAILABLE:
            raw_signals = read_raw_signals_pyslow5(BLOW5_FILE, unique_reads)
            print(f"Successfully read raw signals for {len(raw_signals)} reads")
        else:
            print("pyslow5 not available, trying slow5tools CLI...")
            raw_signals = read_raw_signals_cli(BLOW5_FILE, unique_reads, Path("../output/tmp"))
    except Exception as e:
        print(f"Error reading BLOW5: {e}")
        raw_signals = {}
else:
    raw_signals = {}
    print("No test cases or BLOW5 file not found.")

## 5. Compare Concatenated vs Raw Signals

For each test case:
1. Get the concatenated signal from eventalign
2. Slice the raw signal using `[start_idx:end_idx]`
3. Compare the two arrays

In [ ]:
def compare_signals(concatenated: np.ndarray, raw_sliced: np.ndarray, rtol: float = 1e-5) -> dict:
    """
    Compare concatenated signal with raw sliced signal.
    
    Returns
    -------
    dict with comparison results
    """
    result = {
        'concat_len': len(concatenated),
        'raw_len': len(raw_sliced),
        'length_match': len(concatenated) == len(raw_sliced),
    }
    
    if not result['length_match']:
        result['match'] = False
        result['reason'] = f"Length mismatch: {len(concatenated)} vs {len(raw_sliced)}"
        return result
    
    # Check if values match
    try:
        np.testing.assert_allclose(concatenated, raw_sliced, rtol=rtol)
        result['match'] = True
        result['reason'] = "Arrays are equal within tolerance"
    except AssertionError as e:
        result['match'] = False
        result['reason'] = str(e)
        
        # Compute statistics for debugging
        diff = concatenated - raw_sliced
        result['max_abs_diff'] = float(np.max(np.abs(diff)))
        result['mean_abs_diff'] = float(np.mean(np.abs(diff)))
        result['max_rel_diff'] = float(np.max(np.abs(diff) / (np.abs(raw_sliced) + 1e-10)))
    
    return result


# Compare all selected test cases
results = []

for tc in SELECTED_CASES:
    read_name = tc['read_name']
    if read_name not in raw_signals:
        print(f"WARNING: Raw signal not found for read {read_name}")
        continue
    
    raw = raw_signals[read_name]
    start_idx = tc['start_idx']
    end_idx = tc['end_idx']
    
    # Slice raw signal
    raw_sliced = raw[start_idx:end_idx]
    
    # Compare
    cmp = compare_signals(tc['concatenated_signal'], raw_sliced)
    cmp['position'] = tc['position']
    cmp['read_name'] = read_name
    cmp['start_idx'] = start_idx
    cmp['end_idx'] = end_idx
    cmp['n_events'] = tc['n_events']
    
    results.append(cmp)

# Summary
print(f"\n{'='*80}")
print("VERIFICATION RESULTS")
print(f"{'='*80}\n")

if results:
    n_match = sum(1 for r in results if r['match'])
    n_total = len(results)
    print(f"Matched: {n_match}/{n_total} ({100*n_match/n_total:.1f}%)")
    print()
    
    print(f"{'Position':>10}  {'Read':<25}  {'Events':>6}  {'Concat len':>10}  {'Raw len':>10}  {'Match':>6}")
    print("-" * 85)
    for r in results:
        read_short = r['read_name'][:25]
        print(f"{r['position']:10d}  {read_short:<25}  {r['n_events']:6d}  "
              f"{r['concat_len']:10d}  {r['raw_len']:10d}  {'YES' if r['match'] else 'NO':>6}")
    
    # Show details for failures
    failures = [r for r in results if not r['match']]
    if failures:
        print(f"\n{'='*80}")
        print("FAILURE DETAILS")
        print(f"{'='*80}\n")
        for r in failures:
            print(f"Position {r['position']}, Read {r['read_name']}:")
            print(f"  Reason: {r['reason']}")
            if 'max_abs_diff' in r:
                print(f"  Max abs diff: {r['max_abs_diff']:.6f}")
                print(f"  Mean abs diff: {r['mean_abs_diff']:.6f}")
                print(f"  Max rel diff: {r['max_rel_diff']:.6f}")
            print()
else:
    print("No results to compare.")

## 6. Visualize Comparison

Plot concatenated vs raw signals for visual inspection.

In [ ]:
import matplotlib.pyplot as plt

# Plot first 3 test cases (matched or not)
n_plot = min(3, len(results))

if n_plot > 0:
    fig, axes = plt.subplots(n_plot, 1, figsize=(14, 4 * n_plot), squeeze=False)
    
    for idx, r in enumerate(results[:n_plot]):
        ax = axes[idx, 0]
        
        read_name = r['read_name']
        tc = next(tc for tc in SELECTED_CASES if tc['read_name'] == read_name and tc['position'] == r['position'])
        
        concatenated = tc['concatenated_signal']
        raw = raw_signals[read_name]
        start_idx = r['start_idx']
        end_idx = r['end_idx']
        raw_sliced = raw[start_idx:end_idx]
        
        x_concat = np.arange(len(concatenated))
        x_raw = np.arange(len(raw_sliced))
        
        ax.plot(x_concat, concatenated, 'b-', alpha=0.7, linewidth=1, label='Concatenated (eventalign)')
        ax.plot(x_raw, raw_sliced, 'r--', alpha=0.7, linewidth=1, label='Raw sliced [start_idx:end_idx]')
        
        match_str = "MATCH" if r['match'] else "MISMATCH"
        color = "green" if r['match'] else "red"
        ax.set_title(f"Position {r['position']}, Read {read_name[:30]}... | {match_str}",
                     fontsize=11, fontweight="bold", color=color)
        ax.set_xlabel("Sample index")
        ax.set_ylabel("Signal (pA)")
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No results to plot.")

## 7. Summary

This notebook verifies that the signal concatenate logic in `group_signals_by_position()`
correctly reconstructs the original signal by:

1. Parsing eventalign TSV output with `--signal-index` flag
2. Grouping events by (read_name, position)
3. Sorting events by `start_idx` before concatenation
4. Comparing the concatenated signal with raw signal sliced by `[start_idx:end_idx]`

### Expected Result

If the concatenate logic is correct, the concatenated signal should match the
raw signal slice for all test cases.